# Training CNN Medical Image Segmentation di Google Colab

Notebook ini menjalankan training dari project **MIS·Lab** memakai GPU gratis Colab (jauh lebih cepat dari CPU laptop).

**Alur:**
1. Aktifkan GPU (Runtime > Change runtime type > GPU)
2. Upload project (zip) & mount dataset dari Google Drive
3. Install dependencies
4. Jalankan training
5. Download model hasil (`.h5`) untuk dipakai di web app lokal (VS Code)

## 1. Cek GPU aktif
Pastikan sebelum ini kamu sudah set **Runtime > Change runtime type > Hardware accelerator > GPU (T4)**.

In [ ]:
!nvidia-smi

## 2. Upload project (medseg-app.zip)
Upload file `medseg-app.zip` yang sudah kamu punya (folder project dari laptop) langsung lewat dialog di bawah.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pilih medseg-app.zip dari komputer kamu

import zipfile
for fname in uploaded.keys():
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/')
print('Project sudah diextract ke /content/medseg-app')

## 3. Mount Google Drive (untuk dataset)
Dataset kamu (19.000+ citra) terlalu besar untuk upload manual berkali-kali. Cara paling praktis:
1. Upload folder `dataset` (images/ + masks/) kamu ke Google Drive dulu (lewat browser, drag & drop ke drive.google.com — sekali saja, biar tersimpan permanen)
2. Mount Drive di sini, lalu training langsung baca dari situ

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# GANTI path ini sesuai lokasi folder dataset kamu di Google Drive.
# Contoh kalau kamu taruh di Drive > medseg > dataset:
DATASET_PATH = '/content/drive/MyDrive/medseg/dataset'

import os
print('Isi folder dataset:', os.listdir(DATASET_PATH))

## 4. Install dependencies
Colab sudah punya TensorFlow & numpy bawaan, kita cuma perlu tambahan library kecil.

In [ ]:
!pip install -q opencv-python-headless scikit-image scikit-learn

## 5. Jalankan training
Sama persis seperti perintah di laptop, cuma dijalankan di sini (jauh lebih cepat berkat GPU).

In [ ]:
%cd /content/medseg-app

!python train.py \
  --data_dir "{DATASET_PATH}" \
  --method clahe_he \
  --epochs 30 \
  --max_samples 3616 \
  --out_dir /content/medseg-app/models

## 6. (Opsional) Lanjutkan training kalau ingin lebih banyak epoch
Sama seperti di laptop, pakai `--resume` supaya tidak mulai dari nol lagi.

In [ ]:
!python train.py \
  --data_dir "{DATASET_PATH}" \
  --method clahe_he \
  --epochs 30 \
  --max_samples 3616 \
  --out_dir /content/medseg-app/models \
  --resume

## 7. Download model hasil training
Model `.h5` ini yang nanti kamu taruh ke folder `models/` di project VS Code lokal kamu, supaya web app otomatis pakai model terlatih ini.

In [ ]:
from google.colab import files
files.download('/content/medseg-app/models/clahe_he.h5')
files.download('/content/medseg-app/models/results.json')

## 8. (Opsional) Backup model ke Google Drive juga
Supaya tidak hilang kalau sesi Colab berakhir sebelum sempat download manual.

In [ ]:
import shutil
shutil.copy('/content/medseg-app/models/clahe_he.h5', '/content/drive/MyDrive/medseg/clahe_he.h5')
print('Model dibackup ke Google Drive.')